In [1]:
# Cell 1 — install the two tools we need
!pip -q install datasets sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.7 MB/s eta 0:00:00


In [2]:
# Cell 2 — load the English→Spanish data and see its shape
from datasets import load_dataset

ds = load_dataset("sapienzanlp/ea-mt-benchmark", "en-es")
print(ds)   # shows the splits and how many rows each has

# check which splits actually include the correct Spanish translation (the answer key)
for split in ds:
    ex = ds[split][0]
    has_ref = bool(ex["targets"]) and bool(ex["targets"][0].get("translation"))
    print(f"{split}: {len(ds[split])} rows | has answer key: {has_ref}")

README.md:   0%|          | 0.00/7.38k [00:00<?, ?B/s]

es_ES.jsonl:   0%|          | 0.00/30.8k [00:00<?, ?B/s]

es_ES.jsonl:   0%|          | 0.00/303k [00:00<?, ?B/s]

es_ES.jsonl:   0%|          | 0.00/2.24M [00:00<?, ?B/s]

Generating sample split:   0%|          | 0/72 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/739 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5338 [00:00<?, ? examples/s]

DatasetDict({
    sample: Dataset({
        features: ['id', 'wikidata_id', 'entity_types', 'source', 'targets', 'source_locale', 'target_locale'],
        num_rows: 72
    })
    validation: Dataset({
        features: ['id', 'wikidata_id', 'entity_types', 'source', 'targets', 'source_locale', 'target_locale'],
        num_rows: 739
    })
    test: Dataset({
        features: ['id', 'wikidata_id', 'entity_types', 'source', 'targets', 'source_locale', 'target_locale'],
        num_rows: 5338
    })
})
sample: 72 rows | has answer key: True
validation: 739 rows | has answer key: True
test: 5338 rows | has answer key: True


In [3]:
# Cell 3 — look at 5 real examples: English in, correct Spanish out
for ex in ds["validation"].select(range(5)):
    print("EN    :", ex["source"])
    print("ES    :", ex["targets"][0]["translation"])
    print("entity:", ex["targets"][0]["mention"], "| type:", ex["entity_types"])
    print("-" * 60)

EN    : Where is the Basilica of St. Sernin situated?
ES    : ¿Dónde está la Basílica de San Sernin?
entity: Basílica de San Sernin | type: ['Place of worship']
------------------------------------------------------------
EN    : Where is the Basilica of St. Sernin located?
ES    : ¿Dónde está la Basílica de San Sernin?
entity: Basílica de San Sernin | type: ['Place of worship']
------------------------------------------------------------
EN    : What type of place is the Basilica of St. Sernin?
ES    : ¿Qué tipo de lugar es la Basílica de San Sernin?
entity: Basílica de San Sernin | type: ['Place of worship']
------------------------------------------------------------
EN    : What temperature is best for serving white wine?
ES    : ¿Qué temperatura es la mejor para servir vino blanco?
entity: vino blanco | type: ['Food']
------------------------------------------------------------
EN    : How is white wine different from red wine?
ES    : En qué se diferencia el vino blanco del vino 

In [4]:
# Cell 4 — load NLLB-600M, the model we'll fine-tune
!pip -q install sentencepiece
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "facebook/nllb-200-distilled-600M"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to("cuda")
print("loaded on:", model.device)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

loaded on: cuda:0


In [5]:
# Cell 5 — translate 5 test sentences with the base model
def translate(texts, max_new_tokens=128):
    tok.src_lang = "eng_Latn"
    batch = tok(texts, return_tensors="pt", padding=True, truncation=True).to("cuda")
    out = model.generate(
        **batch,
        forced_bos_token_id=tok.convert_tokens_to_ids("spa_Latn"),
        max_new_tokens=max_new_tokens,
    )
    return tok.batch_decode(out, skip_special_tokens=True)

samples = ds["test"].select(range(5))
preds = translate([ex["source"] for ex in samples])
for ex, p in zip(samples, preds):
    print("EN  :", ex["source"])
    print("GOLD:", ex["targets"][0]["translation"])
    print("NLLB:", p)
    print("-" * 60)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EN  : What kind of artwork is The Signal-Man?
GOLD: ¿Qué tipo de obra artistica es El guardavía?
NLLB: ¿Qué tipo de obra de arte es El Hombre de la Señal?
------------------------------------------------------------
EN  : When was The Signal-Man first published?
GOLD: ¿Cuándo se publicó por primera vez El guardavía?
NLLB: ¿Cuándo se publicó por primera vez El hombre de la señal?
------------------------------------------------------------
EN  : Who is the author of The Signal-Man?
GOLD: ¿Quién es el autor de El guardavía?
NLLB: ¿Quién es el autor de El hombre de la señal?
------------------------------------------------------------
EN  : Who are some of the main actors in the movie About Joan?
GOLD: ¿Quiénes son algunos de los actores principales de la película À propos de Joan?
NLLB: ¿Quiénes son algunos de los principales actores de la película About Joan?
------------------------------------------------------------
EN  : Who directed the 2022 film About Joan?
GOLD: ¿Quién dirigió la

In [6]:
# Cell 6 — baseline BLEU of the base model on a chunk of the test set
import sacrebleu

def translate_batched(texts, bs=16, max_new_tokens=128):
    out = []
    for i in range(0, len(texts), bs):
        out += translate(texts[i:i+bs], max_new_tokens)
    return out

N = 300  # quick estimate first; we run the full 5338 later
subset = ds["test"].select(range(N))
hyps = translate_batched([ex["source"] for ex in subset])

# each example can have up to 3 correct answers — BLEU uses all of them
max_refs = max(len(ex["targets"]) for ex in subset)
refs = [[] for _ in range(max_refs)]
for ex in subset:
    trs = [t["translation"] for t in ex["targets"]]
    for j in range(max_refs):
        refs[j].append(trs[j] if j < len(trs) else trs[0])  # pad short ones

bleu = sacrebleu.corpus_bleu(hyps, refs)
print(f"Base NLLB-600M  |  BLEU on {N} test examples = {bleu.score:.2f}")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Base NLLB-600M  |  BLEU on 300 test examples = 58.02


In [9]:
# Cell 7 — attach LoRA adapters (first remove a conflicting unused library)
!pip -q uninstall -y torchao
!pip -q install peft

from peft import LoraConfig, get_peft_model
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    task_type="SEQ_2_SEQ_LM",
)
peft_model = get_peft_model(model, lora)
peft_model.print_trainable_parameters()

trainable params: 4,718,592 || all params: 619,792,384 || trainable%: 0.7613


In [10]:
# Cell 8 — turn the 739 training examples into model-ready form
def preprocess(batch):
    tok.src_lang = "eng_Latn"
    tok.tgt_lang = "spa_Latn"
    inputs = batch["source"]
    targets = [t[0]["translation"] for t in batch["targets"]]  # first correct answer
    enc = tok(inputs, max_length=128, truncation=True)
    labels = tok(text_target=targets, max_length=128, truncation=True)
    enc["labels"] = labels["input_ids"]
    return enc

train_ds = ds["validation"].map(
    preprocess, batched=True, remove_columns=ds["validation"].column_names
)
print(train_ds)

Map:   0%|          | 0/739 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 739
})


In [12]:
# Cell 9 — fine-tune (the actual training run)
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(tok, model=peft_model)
args = Seq2SeqTrainingArguments(
    output_dir="nllb-es-lora",
    per_device_train_batch_size=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=20,
    save_strategy="no",
    fp16=True,
    report_to="none",
)
trainer = Seq2SeqTrainer(
    model=peft_model, args=args, train_dataset=train_ds,
    data_collator=collator, processing_class=tok,   # renamed from tokenizer
)
trainer.train()

Step,Training Loss
20,1.216170
40,0.945820
60,0.778253
80,0.822964
100,0.738855
120,0.827118
140,0.648794
160,0.666468
180,0.756322
200,0.703307


TrainOutput(global_step=279, training_loss=0.7607906334716359, metrics={'train_runtime': 62.2506, 'train_samples_per_second': 35.614, 'train_steps_per_second': 4.482, 'total_flos': 104807806156800.0, 'train_loss': 0.7607906334716359, 'epoch': 3.0})

In [13]:
# Cell 10 — fine-tuned model's BLEU on the SAME 300 test examples
peft_model.eval()

def translate_ft(texts, max_new_tokens=128):
    tok.src_lang = "eng_Latn"
    batch = tok(texts, return_tensors="pt", padding=True, truncation=True).to("cuda")
    out = peft_model.generate(
        **batch,
        forced_bos_token_id=tok.convert_tokens_to_ids("spa_Latn"),
        max_new_tokens=max_new_tokens,
    )
    return tok.batch_decode(out, skip_special_tokens=True)

srcs = [ex["source"] for ex in subset]     # same 300 test rows from Cell 6
hyps_ft = []
for i in range(0, len(srcs), 16):
    hyps_ft += translate_ft(srcs[i:i+16])

bleu_ft = sacrebleu.corpus_bleu(hyps_ft, refs)   # same references from Cell 6
print(f"Fine-tuned + LoRA | BLEU on {N} test examples = {bleu_ft.score:.2f}")
print(f"baseline was 58.02  |  paper's Spanish = 59.14")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Fine-tuned + LoRA | BLEU on 300 test examples = 64.75
baseline was 58.02  |  paper's Spanish = 59.14


In [14]:
# Cell 11 — fair comparison on the FULL test set (5338 examples)
import sacrebleu
peft_model.eval()

full = ds["test"]
srcs_full = [ex["source"] for ex in full]

max_refs = max(len(ex["targets"]) for ex in full)
refs_full = [[] for _ in range(max_refs)]
for ex in full:
    trs = [t["translation"] for t in ex["targets"]]
    for j in range(max_refs):
        refs_full[j].append(trs[j] if j < len(trs) else trs[0])

def gen_all():
    hyps = []
    for i in range(0, len(srcs_full), 16):
        tok.src_lang = "eng_Latn"
        b = tok(srcs_full[i:i+16], return_tensors="pt", padding=True, truncation=True).to("cuda")
        g = peft_model.generate(**b,
              forced_bos_token_id=tok.convert_tokens_to_ids("spa_Latn"),
              max_new_tokens=128)
        hyps += tok.batch_decode(g, skip_special_tokens=True)
    return hyps

# fine-tuned (LoRA on)
ft = sacrebleu.corpus_bleu(gen_all(), refs_full).score
# base (LoRA off)
with peft_model.disable_adapter():
    base = sacrebleu.corpus_bleu(gen_all(), refs_full).score

print(f"FULL TEST (5338)  |  base = {base:.2f}   fine-tuned = {ft:.2f}   (paper Spanish = 59.14)")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

FULL TEST (5338)  |  base = 60.22   fine-tuned = 65.25   (paper Spanish = 59.14)


In [15]:
# Cell 12 — entity accuracy: does the model get the NAME right? (300-example estimate)
def entity_hits(hyps, dataset):
    hits = 0
    for h, ex in zip(hyps, dataset):
        mentions = [t["mention"] for t in ex["targets"]]
        if any(m and m in h for m in mentions):
            hits += 1
    return 100 * hits / len(dataset)

subset = ds["test"].select(range(300))
srcs = [ex["source"] for ex in subset]

def gen_subset():
    hs = []
    for i in range(0, len(srcs), 16):
        tok.src_lang = "eng_Latn"
        b = tok(srcs[i:i+16], return_tensors="pt", padding=True, truncation=True).to("cuda")
        g = peft_model.generate(**b,
              forced_bos_token_id=tok.convert_tokens_to_ids("spa_Latn"),
              max_new_tokens=128)
        hs += tok.batch_decode(g, skip_special_tokens=True)
    return hs

ft_preds = gen_subset()
with peft_model.disable_adapter():
    base_preds = gen_subset()

print(f"Entity accuracy (300 test) | base = {entity_hits(base_preds, subset):.1f}%"
      f"   fine-tuned = {entity_hits(ft_preds, subset):.1f}%")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Entity accuracy (300 test) | base = 19.7%   fine-tuned = 23.7%
